# Financial News Sentiment Prediction using Deep Learning
Bearish / Bullish / Neutral classification on financial tweets, comparing Simple RNN, LSTM, and GRU baselines.

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import re
import pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import confusion_matrix, f1_score, classification_report
import copy

## 2. Load Data

In [2]:
df_train = pd.read_csv('sent_train.csv')
df_valid = pd.read_csv('sent_valid.csv')
print(df_train.shape, df_valid.shape)
df_train.head()

(9543, 2) (2388, 2)


,text,label
0,$BYND - JPMorgan reels in expectations on Beyo...,0
1,$CCL $RCL - Nomura points to bookings weakness...,0
2,"$CX - Cemex cut at Credit Suisse, J.P. Morgan ...",0
3,$ESS: BTIG Research cuts to Neutral https://t....,0
4,$FNKO - Funko slides after Piper Jaffray PT cu...,0


## 3. Data Quality Checks
Check for nulls, duplicates, and class distribution before doing anything else.

In [3]:
print('Train nulls:\n', df_train.isnull().sum())
print('Valid nulls:\n', df_valid.isnull().sum())
print('Train duplicates:', df_train.duplicated().sum())
print('Valid duplicates:', df_valid.duplicated().sum())

Train nulls:
 text     0
label    0
dtype: int64
Valid nulls:
 text     0
label    0
dtype: int64
Train duplicates: 0
Valid duplicates: 0


In [4]:
print('Train class distribution:')
print(df_train['label'].value_counts())
print('\nValid class distribution:')
print(df_valid['label'].value_counts())

Train class distribution:
label
2    6178
1    1923
0    1442
Name: count, dtype: int64

Valid class distribution:
label
2    1566
1     475
0     347
Name: count, dtype: int64


**Finding:** the dataset is imbalanced (~65% Neutral, ~20% Bullish, ~15% Bearish). This is addressed later using class-weighted cross-entropy loss rather than oversampling/undersampling/SMOTE, since resampling techniques risk distorting raw text data.

## 4. Text Cleaning
Removes URLs, mentions, quotes, punctuation, and separator-style hyphens (e.g. `"$ally - ally"`), while preserving stock tickers (`$AAPL`) and negative numbers (`-5%`). Collapses extra whitespace and lowercases.

In [5]:
def clean_text(text):
    text = re.sub(r"https?://\S+", "", text)          # remove URLs
    text = re.sub(r"@\w+", "", text)                    # remove mentions
    text = re.sub('"', "", text)                         # remove quotes
    text = re.sub("[.,:;!?]", "", text)                 # remove punctuation
    text = re.sub(" - ", " ", text)                      # remove separator hyphens
    text = re.sub(r"\s+", " ", text)                    # collapse extra spaces
    text = text.strip()
    text = text.lower()
    return text

df_train['text'] = df_train['text'].apply(clean_text)
df_valid['text'] = df_valid['text'].apply(clean_text)
df_train.head()

,text,label
0,$bynd jpmorgan reels in expectations on beyond...,0
1,$ccl $rcl nomura points to bookings weakness a...,0
2,$cx cemex cut at credit suisse jp morgan on we...,0
3,$ess btig research cuts to neutral,0
4,$fnko funko slides after piper jaffray pt cut,0


## 5. Tokenization

In [6]:
df_train['tokens'] = df_train['text'].apply(lambda x: x.split())
df_valid['tokens'] = df_valid['text'].apply(lambda x: x.split())

## 6. Vocabulary Building
Built **exclusively from the training set** to avoid data leakage into validation.

In [7]:
all_words = []
for i in df_train['tokens']:
    for j in i:
        all_words.append(j)
vocab = set(all_words)
print('Vocabulary size (unique words):', len(vocab))

Vocabulary size (unique words): 19501


In [8]:
word2idx = {"<PAD>": 0, "<UNK>": 1}
for index, word in enumerate(vocab, 2):
    word2idx[word] = index
print('Total vocab size (incl. PAD/UNK):', len(word2idx))

Total vocab size (incl. PAD/UNK): 19503


## 7. Encoding (words → numbers)
Unseen words (e.g. present only in validation) safely fall back to `<UNK>` instead of raising an error.

In [9]:
def encode_tokens(tokens):
    tweet = []
    for i in tokens:
        tweet.append(word2idx.get(i, 1))
    return tweet

df_train['encode_text'] = df_train['tokens'].apply(encode_tokens)
df_valid['encode_text'] = df_valid['tokens'].apply(encode_tokens)

In [10]:
# sanity check: how much of validation is unseen (<UNK>) relative to training vocab
unk_count = 0
for i in df_valid['encode_text']:
    for j in i:
        if j == 1:
            unk_count += 1
total_words = sum(len(i) for i in df_valid['encode_text'])
print('Percent unknown in validation:', round(unk_count / total_words * 100, 2), '%')

Percent unknown in validation: 11.28 %


## 8. Remove Empty Rows
A handful of training tweets became empty after cleaning (e.g. URL-only tweets) and carry no signal.

In [11]:
df_train['tweet_length'] = df_train['tokens'].apply(len)
print('Empty rows found:', (df_train['tweet_length'] == 0).sum())
df_train = df_train[df_train['tweet_length'] != 0]
print('Train shape after dropping empty rows:', df_train.shape)

Empty rows found: 6
Train shape after dropping empty rows: (9537, 5)


## 9. Padding / Truncation
Sequences are padded/truncated to a fixed length of 20 tokens — the 95th percentile of training tweet lengths, chosen to balance information loss against wasted padding.

In [12]:
def pad_sequence(tokens, max_len=20):
    tokens = tokens.copy()
    if len(tokens) < max_len:
        for i in range(max_len - len(tokens)):
            tokens.append(0)
        return tokens
    elif len(tokens) > max_len:
        return tokens[:max_len]
    else:
        return tokens

df_train['pad'] = df_train['encode_text'].apply(pad_sequence)
df_valid['pad'] = df_valid['encode_text'].apply(pad_sequence)

## 10. Convert to PyTorch Tensors

In [13]:
X_train = torch.tensor(df_train['pad'].tolist())
X_valid = torch.tensor(df_valid['pad'].tolist())
y_train = torch.tensor(df_train['label'].tolist())
y_valid = torch.tensor(df_valid['label'].tolist())
print(X_train.shape, X_valid.shape, y_train.shape, y_valid.shape)

torch.Size([9537, 20]) torch.Size([2388, 20]) torch.Size([9537]) torch.Size([2388])


## 11. Class Weights, Loss Function, and Batching

In [14]:
class_weights = torch.tensor([2.2, 1.65, 0.5])  # Bearish, Bullish, Neutral
criterion = nn.CrossEntropyLoss(weight=class_weights)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

## 12. Model Architectures
All three models share the same Embedding (vocab size 19,503, embedding dim 50) and Linear output layer (hidden size 64 → 3 classes), differing only in the recurrent layer.

In [15]:
vocab_size = len(word2idx)
embedding_dim = 50
hidden_size = 64
print('vocab_size:', vocab_size)

vocab_size: 19503


In [16]:
class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 3)

    def forward(self, x):
        embedded = self.embedding(x)
        output, hidden = self.rnn(embedded)
        hidden = hidden.squeeze(0)
        return self.fc(hidden)

In [17]:
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 3)

    def forward(self, x):
        embedded = self.embedding(x)
        output, (hidden, cell) = self.lstm(embedded)
        hidden = hidden.squeeze(0)
        return self.fc(hidden)

In [18]:
class SentimentGRU(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.gru = nn.GRU(embedding_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 3)

    def forward(self, x):
        embedded = self.embedding(x)
        output, hidden = self.gru(embedded)
        hidden = hidden.squeeze(0)
        return self.fc(hidden)

## 13. Training Function (with Early Stopping)
Trains for up to `max_epochs`, checking validation accuracy after every epoch. Keeps a deep copy of the best-performing weights seen so far, and stops automatically once validation accuracy fails to improve for `patience` consecutive epochs — preventing the overfitting seen in early manual experiments.

In [19]:
def train_model(model, train_loader, X_valid, y_valid, criterion, max_epochs=50, patience=3):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    best_acc = 0
    best_state = None
    patience_counter = 0

    for epoch in range(max_epochs):
        model.train()
        for batch_X, batch_y in train_loader:
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

        model.eval()
        with torch.no_grad():
            val_outputs = model(X_valid)
        val_preds = val_outputs.argmax(dim=1)
        val_acc = (val_preds == y_valid).sum().item() / len(y_valid)

        print(f'Epoch {epoch}: train_loss={loss.item():.4f}, val_acc={val_acc:.4f}')

        if val_acc > best_acc:
            best_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping at epoch {epoch} (best val_acc={best_acc:.4f})')
                break

    model.load_state_dict(best_state)
    return model, best_acc

## 14. Evaluation Helper

In [20]:
def evaluate(model, X_valid, y_valid, name):
    model.eval()
    with torch.no_grad():
        val_outputs = model(X_valid)
    predictions = val_outputs.argmax(dim=1)
    y_true = y_valid.tolist()
    y_pred = predictions.tolist()
    print(f'--- {name} ---')
    print(confusion_matrix(y_true, y_pred))
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    print('Macro F1:', macro_f1)
    print(classification_report(y_true, y_pred, target_names=['Bearish', 'Bullish', 'Neutral']))
    return macro_f1

## 15. Train &amp; Evaluate: Simple RNN

In [21]:
model_rnn = SentimentRNN(vocab_size, embedding_dim, hidden_size)
model_rnn, rnn_acc = train_model(model_rnn, train_loader, X_valid, y_valid, criterion)
rnn_f1 = evaluate(model_rnn, X_valid, y_valid, 'Simple RNN')

Epoch 0: train_loss=0.9914, val_acc=0.1692
Epoch 1: train_loss=1.2161, val_acc=0.2265
Epoch 2: train_loss=1.2463, val_acc=0.2073
Epoch 3: train_loss=1.0306, val_acc=0.3844
Epoch 4: train_loss=1.4336, val_acc=0.3501
Epoch 5: train_loss=0.3540, val_acc=0.3220
Epoch 6: train_loss=0.9706, val_acc=0.3501
Early stopping at epoch 6 (best val_acc=0.3844)
--- Simple RNN ---
[[ 65 140 142]
 [ 82 201 192]
 [247 667 652]]
Macro F1: 0.3191608447900875
              precision    recall  f1-score   support

     Bearish       0.16      0.19      0.18       347
     Bullish       0.20      0.42      0.27       475
     Neutral       0.66      0.42      0.51      1566

    accuracy                           0.38      2388
   macro avg       0.34      0.34      0.32      2388
weighted avg       0.50      0.38      0.41      2388



## 16. Train &amp; Evaluate: LSTM

In [22]:
model_lstm = SentimentLSTM(vocab_size, embedding_dim, hidden_size)
model_lstm, lstm_acc = train_model(model_lstm, train_loader, X_valid, y_valid, criterion)
lstm_f1 = evaluate(model_lstm, X_valid, y_valid, 'LSTM')

Epoch 0: train_loss=1.1066, val_acc=0.2391
Epoch 1: train_loss=1.0674, val_acc=0.6558
Epoch 2: train_loss=0.5394, val_acc=0.6055
Epoch 3: train_loss=0.4367, val_acc=0.6709
Epoch 4: train_loss=0.1427, val_acc=0.6893
Epoch 5: train_loss=0.5239, val_acc=0.6943
Epoch 6: train_loss=0.1886, val_acc=0.7127
Epoch 7: train_loss=0.0069, val_acc=0.7224
Epoch 8: train_loss=0.0058, val_acc=0.7466
Epoch 9: train_loss=0.0101, val_acc=0.7538
Epoch 10: train_loss=0.0227, val_acc=0.7517
Epoch 11: train_loss=0.0035, val_acc=0.7554
Epoch 12: train_loss=0.0665, val_acc=0.7617
Epoch 13: train_loss=0.0614, val_acc=0.7651
Epoch 14: train_loss=0.1013, val_acc=0.7496
Epoch 15: train_loss=0.0035, val_acc=0.7684
Epoch 16: train_loss=0.0014, val_acc=0.7684
Epoch 17: train_loss=0.0008, val_acc=0.7534
Epoch 18: train_loss=0.0144, val_acc=0.7538
Early stopping at epoch 18 (best val_acc=0.7684)
--- LSTM ---
[[ 169   62  116]
 [  70  267  138]
 [  88   79 1399]]
Macro F1: 0.6584847443481124
              precision    r

## 17. Train &amp; Evaluate: GRU

In [23]:
model_gru = SentimentGRU(vocab_size, embedding_dim, hidden_size)
model_gru, gru_acc = train_model(model_gru, train_loader, X_valid, y_valid, criterion)
gru_f1 = evaluate(model_gru, X_valid, y_valid, 'GRU')

Epoch 0: train_loss=1.1756, val_acc=0.3346
Epoch 1: train_loss=0.5985, val_acc=0.6461
Epoch 2: train_loss=0.2356, val_acc=0.6491
Epoch 3: train_loss=1.0127, val_acc=0.5771
Epoch 4: train_loss=0.8902, val_acc=0.6181
Epoch 5: train_loss=0.1237, val_acc=0.6240
Early stopping at epoch 5 (best val_acc=0.6491)
--- GRU ---
[[ 142  113   92]
 [ 123  261   91]
 [ 286  133 1147]]
Macro F1: 0.5466512172740688
              precision    recall  f1-score   support

     Bearish       0.26      0.41      0.32       347
     Bullish       0.51      0.55      0.53       475
     Neutral       0.86      0.73      0.79      1566

    accuracy                           0.65      2388
   macro avg       0.54      0.56      0.55      2388
weighted avg       0.71      0.65      0.67      2388



## 18. Model Comparison Summary

In [24]:
summary = pd.DataFrame({
    'Model': ['Simple RNN', 'LSTM', 'GRU'],
    'Accuracy': [rnn_acc, lstm_acc, gru_acc],
    'Macro F1': [rnn_f1, lstm_f1, gru_f1]
})
summary

,Model,Accuracy,Macro F1
0,Simple RNN,0.384422,0.319161
1,LSTM,0.768425,0.658485
2,GRU,0.649079,0.546651


**GRU achieved the best accuracy and macro F1** in this run using consistent, automated early stopping across all three architectures, and was selected as the deployed model for the Streamlit dashboard.

## 19. Save Best Model (GRU) &amp; Vocabulary
Saved directly into the project folder so the Streamlit app (`app.py`) can load them without any path ambiguity.

In [28]:
torch.save(model_gru.state_dict(), 'model.pth')

with open('word2idx.pkl', 'wb') as f:
    pickle.dump(word2idx, f)

print('Saved model.pth and word2idx.pkl')

Saved model.pth and word2idx.pkl


## 20. Error Analysis
Two notable failure modes investigated during manual testing of the Streamlit app.

**(a) Label ambiguity for "crash":** the word appears almost as often in Neutral-labeled training tweets as Bearish ones, so a confident Neutral prediction for standalone phrases like "stock crash" reflects genuine ambiguity in the training data rather than a model error.

In [26]:
crash_tweets = df_train[df_train['text'].str.contains('crash')]
print(crash_tweets[['text', 'label']])
print('\nLabel counts among crash-containing tweets:')
print(crash_tweets['label'].value_counts())

                                                   text  label
1823  report emirates pilots unaware engines idle in...      2
3015  midstream prices crash maybe we have seen this...      0
3028  the crash in energy prices has hit norway euro...      0
3038  us gasoline crashes to 50c lowest since 2001 a...      0
3100  investment in canada's oil-sands is on track t...      1
3147  saudi arabia russia and other large oil produc...      2
3693          hsbc says british pound may soar or crash      2
4983  power rankings retail roller coaster crypto cr...      2
5078    so you crashed your $3 million bugatti now what      2
5749  dot watchdog probes pilot training requirement...      0
5752  faa predicted boeing 737 max facing more delay...      0
5783  us safety board chair criticizes uber for 2018...      0
5892  transportation department ig to audit faa pilo...      2
5893  transportation department ig to audit faa pilo...      2
6190  china's skyscraper boom comes crashing down am...

**(b) Truncation sensitivity:** for tweets longer than 20 tokens, truncation can remove decisive sentiment cues near the end of the text.

In [27]:
test_tweet = "BREAKING: $AAPL reports Q2 revenue of $94.8B, beating analyst estimates by 3%. "\
             "iPhone sales remain the key driver. Shares up 4% in after-hours trading"
cleaned = clean_text(test_tweet)
tokens = cleaned.split()
print('Token count:', len(tokens))
print('Tokens kept after truncation to 20:', tokens[:20])

Token count: 24
Tokens kept after truncation to 20: ['breaking', '$aapl', 'reports', 'q2', 'revenue', 'of', '$948b', 'beating', 'analyst', 'estimates', 'by', '3%', 'iphone', 'sales', 'remain', 'the', 'key', 'driver', 'shares', 'up']
